In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.pipe;

-- Control Table
CREATE TABLE IF NOT EXISTS workspace.pipe.fraud_pipeline_run_log (
    run_id BIGINT,
    run_timestamp TIMESTAMP,
    status STRING,
    queries_executed INT
);

In [0]:
%sql
-- 1. Empleados con mas voids sospechosos (quantity < 1.0 = void total o parcial, sold_price > 0)
--    Cruzado con locacion para identificar en que sucursal trabaja cada empleado
--    Incluye tasa de voids (voids / (voids + vendidos)) para contextualizar el riesgo
CREATE OR REPLACE TABLE workspace.pipe.fraud_voids_by_employee AS
WITH void_stats AS (
    SELECT employee_id, location_id,
           COUNT(*) AS items_voided,
           ROUND(SUM(sold_price), 2) AS valor_voided,
           ROUND(AVG(sold_price), 2) AS precio_prom_void,
           COUNT(DISTINCT check_id) AS checks_afectados
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity < 1.0 AND sold_price > 0
    GROUP BY employee_id, location_id
),
total_stats AS (
    SELECT employee_id, location_id,
           COUNT(*) AS total_items_vendidos,
           ROUND(SUM(sold_price), 2) AS total_vendido
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity >= 1.0
    GROUP BY employee_id, location_id
)
SELECT v.employee_id,
       v.location_id,
       l.location_name,
       l.status AS location_status,
       v.items_voided,
       v.valor_voided,
       v.precio_prom_void,
       v.checks_afectados,
       COALESCE(t.total_items_vendidos, 0) AS total_items_vendidos,
       COALESCE(t.total_vendido, 0) AS total_vendido,
       ROUND(v.items_voided * 100.0 / NULLIF(v.items_voided + COALESCE(t.total_items_vendidos, 0), 0), 2) AS pct_void_rate
FROM void_stats v
LEFT JOIN total_stats t
    ON v.employee_id = t.employee_id AND v.location_id = t.location_id
LEFT JOIN workspace.applebees_clean.applebees_locations l
    ON v.location_id = l.location_id
ORDER BY v.valor_voided DESC;

In [0]:
%sql
-- 2. Voids por location (solo locaciones Active para consistencia con Query 7)
--    Incluye tasa de voids relativa al volumen total de la locacion
CREATE OR REPLACE TABLE workspace.pipe.fraud_voids_by_location AS
WITH void_stats AS (
    SELECT c.location_id,
           COUNT(*) AS items_voided,
           ROUND(SUM(c.sold_price), 2) AS valor_voided,
           COUNT(DISTINCT c.check_id) AS checks_afectados
    FROM workspace.applebees_clean.applebees_checks_items c
    WHERE c.quantity < 1.0 AND c.sold_price > 0
    GROUP BY c.location_id
),
total_stats AS (
    SELECT location_id,
           COUNT(*) AS total_items,
           ROUND(SUM(sold_price), 2) AS total_ventas
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity >= 1.0
    GROUP BY location_id
)
SELECT v.location_id,
       l.location_name,
       l.status,
       v.items_voided,
       v.valor_voided,
       v.checks_afectados,
       COALESCE(t.total_items, 0) AS total_items_vendidos,
       COALESCE(t.total_ventas, 0) AS total_ventas,
       ROUND(v.items_voided * 100.0 / NULLIF(v.items_voided + COALESCE(t.total_items, 0), 0), 2) AS pct_void_rate
FROM void_stats v
LEFT JOIN total_stats t ON v.location_id = t.location_id
LEFT JOIN workspace.applebees_clean.applebees_locations l ON v.location_id = l.location_id
WHERE l.status = 'Active'
ORDER BY v.valor_voided DESC;

In [0]:
%sql
-- 3. Voids por hora del dia, con tasa relativa al volumen de ventas por hora
CREATE OR REPLACE TABLE workspace.pipe.fraud_voids_by_hour AS
WITH void_stats AS (
    SELECT HOUR(order_time) AS hora,
           COUNT(*) AS items_voided,
           ROUND(SUM(sold_price), 2) AS valor_voided,
           COUNT(DISTINCT check_id) AS checks_afectados
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity < 1.0 AND sold_price > 0
    GROUP BY HOUR(order_time)
),
total_stats AS (
    SELECT HOUR(order_time) AS hora,
           COUNT(*) AS total_items,
           ROUND(SUM(sold_price), 2) AS total_ventas
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity >= 1.0
    GROUP BY HOUR(order_time)
)
SELECT v.hora,
       v.items_voided,
       v.valor_voided,
       v.checks_afectados,
       COALESCE(t.total_items, 0) AS total_items_hora,
       COALESCE(t.total_ventas, 0) AS total_ventas_hora,
       ROUND(v.items_voided * 100.0 / NULLIF(v.items_voided + COALESCE(t.total_items, 0), 0), 2) AS pct_void_rate
FROM void_stats v
LEFT JOIN total_stats t ON v.hora = t.hora
ORDER BY v.hora;

In [0]:
%sql
-- 4. Top items voided, con ratio void/vendido para identificar items propensos a fraude
CREATE OR REPLACE TABLE workspace.pipe.fraud_top_voided_items AS
WITH void_stats AS (
    SELECT item_name, item_subcategory,
           COUNT(*) AS veces_voided,
           ROUND(SUM(sold_price), 2) AS valor_voided,
           ROUND(AVG(sold_price), 2) AS precio_prom
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity < 1.0 AND sold_price > 0
    GROUP BY item_name, item_subcategory
),
sold_stats AS (
    SELECT item_name, item_subcategory,
           COUNT(*) AS veces_vendido,
           ROUND(SUM(sold_price), 2) AS valor_vendido
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity >= 1.0
    GROUP BY item_name, item_subcategory
)
SELECT v.item_name, v.item_subcategory,
       v.veces_voided, v.valor_voided, v.precio_prom,
       COALESCE(s.veces_vendido, 0) AS veces_vendido,
       COALESCE(s.valor_vendido, 0) AS valor_vendido,
       ROUND(v.veces_voided * 100.0 / NULLIF(v.veces_voided + COALESCE(s.veces_vendido, 0), 0), 2) AS pct_void_rate
FROM void_stats v
LEFT JOIN sold_stats s ON v.item_name = s.item_name AND v.item_subcategory = s.item_subcategory
ORDER BY v.veces_voided DESC;

In [0]:
%sql
-- 5. Checks con mas voids (umbral reducido a 3+ items voided por el tamano del dataset)
CREATE OR REPLACE TABLE workspace.pipe.fraud_checks_most_voids AS
SELECT check_id, location_id, employee_id,
       COUNT(*) AS items_voided,
       ROUND(SUM(sold_price), 2) AS valor_voided,
       MIN(order_time) AS primera_orden,
       MAX(order_time) AS ultima_orden
FROM workspace.applebees_clean.applebees_checks_items
WHERE quantity < 1.0 AND sold_price > 0
GROUP BY check_id, location_id, employee_id
HAVING COUNT(*) >= 3
ORDER BY items_voided DESC;

In [0]:
%sql
-- 6. Checks voided vs vendidos, con tasa de void % y flag de patron espejo
--    Patron espejo = mismo item aparece voided Y vendido en el mismo check (duplicacion sospechosa)
--    OPTIMIZACION (28.5M filas):
--    1. Self-join eliminado -> GROUP BY + conditional aggregation (O(n) vs O(n²))
--    2. IN subquery eliminado -> HAVING (evita scan adicional de la tabla completa)
CREATE OR REPLACE TABLE workspace.pipe.fraud_voids_vs_sold AS
WITH item_mirror AS (
    -- Deteccion de patron espejo sin self-join: agrupar por (check_id, item_name) y verificar si el mismo item aparece voided Y vendido en el mismo check
    SELECT check_id, COUNT(*) AS items_espejo
    FROM (
        SELECT check_id, item_name,
               SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) AS voided_count,
               SUM(CASE WHEN quantity >= 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) AS sold_count
        FROM workspace.applebees_clean.applebees_checks_items
        GROUP BY check_id, item_name
    ) x
    WHERE voided_count > 0 AND sold_count > 0
    GROUP BY check_id
),
check_items AS (
    -- Agregacion por check: voids vs vendidos
    -- HAVING reemplaza al IN subquery para filtrar solo checks con voids
    SELECT check_id, location_id, employee_id,
           SUM(CASE WHEN quantity < 1.0 THEN 1 ELSE 0 END) AS items_voided,
           SUM(CASE WHEN quantity >= 1.0 THEN 1 ELSE 0 END) AS items_vendidos,
           ROUND(SUM(CASE WHEN quantity < 1.0 THEN sold_price ELSE 0 END), 2) AS valor_voided,
           ROUND(SUM(CASE WHEN quantity >= 1.0 THEN sold_price ELSE 0 END), 2) AS valor_vendido,
           COUNT(*) AS total_items
    FROM workspace.applebees_clean.applebees_checks_items
    GROUP BY check_id, location_id, employee_id
    HAVING SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) > 0
)
SELECT c.check_id, c.location_id, c.employee_id,
       c.items_voided,
       c.items_vendidos,
       c.valor_voided,
       c.valor_vendido,
       ROUND(c.items_voided * 100.0 / c.total_items, 2) AS pct_void_rate,
       COALESCE(m.items_espejo, 0) AS items_espejo,
       CASE WHEN COALESCE(m.items_espejo, 0) > 0 THEN 1 ELSE 0 END AS flag_patron_espejo
FROM check_items c
LEFT JOIN item_mirror m ON c.check_id = m.check_id
ORDER BY pct_void_rate DESC, c.items_voided DESC;

In [0]:
%sql
-- 7. Items gratis (N/C = No Charge) con proporcion estadisticamente inusual (IQR)
--    Reemplaza z-score por IQR (Interquartile Range) -- no asume distribucion normal
--    IQR es robusto ante outliers y estable con muestras pequenas
--    Umbral: Q3 + 1.5 * IQR (regla estandar de Tukey)
CREATE OR REPLACE TABLE workspace.pipe.fraud_free_items_suspicious AS
WITH check_stats AS (
    SELECT
        ci.check_id,
        ci.location_id,
        COUNT(*) AS total_items,
        SUM(
            CASE
                WHEN UPPER(i.item_name) LIKE '%N/C%'
                THEN 1
                ELSE 0
            END
        ) AS free_items
    FROM workspace.applebees_clean.applebees_checks_items ci
    JOIN workspace.applebees_clean.applebees_items i
        ON ci.item_id = i.item_id
    GROUP BY ci.check_id, ci.location_id
),
check_pct AS (
    SELECT
        cs.check_id,
        l.location_name,
        l.status AS location_status,
        cs.total_items,
        cs.free_items,
        ROUND(cs.free_items * 100.0 / cs.total_items, 2) AS pct_free_items
    FROM check_stats cs
    JOIN workspace.applebees_clean.applebees_locations l
        ON cs.location_id = l.location_id
    WHERE l.status = 'Active'
      AND cs.free_items >= 3
),
iqr_stats AS (
    SELECT
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY pct_free_items) AS q1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY pct_free_items) AS q3
    FROM check_pct
)
SELECT
    cp.check_id,
    cp.location_name,
    cp.location_status,
    cp.total_items,
    cp.free_items,
    cp.pct_free_items,
    ROUND(i.q1, 2) AS q1_free,
    ROUND(i.q3, 2) AS q3_free,
    ROUND(i.q3 - i.q1, 2) AS iqr_free,
    ROUND(i.q3 + 1.5 * (i.q3 - i.q1), 2) AS umbral_iqr
FROM check_pct cp
CROSS JOIN iqr_stats i
WHERE cp.pct_free_items > i.q3 + 1.5 * (i.q3 - i.q1)
ORDER BY cp.free_items DESC, cp.pct_free_items DESC;

In [0]:
%sql
-- 8. Resumen semanal de fraude: agrega metricas clave por semana
--    Permite detectar tendencias y comparar semanas entre si a medida que se cargan mas dias
CREATE OR REPLACE TABLE workspace.pipe.fraud_weekly_summary AS
WITH weekly_voids AS (
    SELECT
        DATE_TRUNC('week', CAST(business_date AS DATE)) AS semana_inicio,
        COUNT(*) AS total_items_voided,
        COUNT(DISTINCT check_id) AS total_checks_afectados,
        COUNT(DISTINCT employee_id) AS empleados_con_voids,
        COUNT(DISTINCT location_id) AS locaciones_con_voids,
        ROUND(SUM(sold_price), 2) AS valor_total_voided,
        ROUND(AVG(sold_price), 2) AS valor_prom_void,
        ROUND(MAX(sold_price), 2) AS valor_max_void
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity < 1.0 AND sold_price > 0
    GROUP BY DATE_TRUNC('week', CAST(business_date AS DATE))
),
weekly_sales AS (
    SELECT
        DATE_TRUNC('week', CAST(business_date AS DATE)) AS semana_inicio,
        COUNT(*) AS total_items_vendidos,
        ROUND(SUM(sold_price), 2) AS total_ventas,
        COUNT(DISTINCT check_id) AS total_checks,
        COUNT(DISTINCT location_id) AS locaciones_activas
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE quantity >= 1.0
    GROUP BY DATE_TRUNC('week', CAST(business_date AS DATE))
),
top_employee AS (
    SELECT semana_inicio, employee_id, location_id, valor_voided
    FROM (
        SELECT
            DATE_TRUNC('week', CAST(business_date AS DATE)) AS semana_inicio,
            employee_id,
            location_id,
            ROUND(SUM(sold_price), 2) AS valor_voided,
            ROW_NUMBER() OVER (
                PARTITION BY DATE_TRUNC('week', CAST(business_date AS DATE))
                ORDER BY SUM(sold_price) DESC
            ) AS rn
        FROM workspace.applebees_clean.applebees_checks_items
        WHERE quantity < 1.0 AND sold_price > 0
        GROUP BY DATE_TRUNC('week', CAST(business_date AS DATE)), employee_id, location_id
    ) t
    WHERE rn = 1
),
top_location AS (
    SELECT semana_inicio, location_id, valor_voided
    FROM (
        SELECT
            DATE_TRUNC('week', CAST(business_date AS DATE)) AS semana_inicio,
            location_id,
            ROUND(SUM(sold_price), 2) AS valor_voided,
            ROW_NUMBER() OVER (
                PARTITION BY DATE_TRUNC('week', CAST(business_date AS DATE))
                ORDER BY SUM(sold_price) DESC
            ) AS rn
        FROM workspace.applebees_clean.applebees_checks_items
        WHERE quantity < 1.0 AND sold_price > 0
        GROUP BY DATE_TRUNC('week', CAST(business_date AS DATE)), location_id
    ) t
    WHERE rn = 1
)
SELECT
    v.semana_inicio,
    DATE_ADD(v.semana_inicio, 6) AS semana_fin,
    v.total_items_voided,
    v.total_checks_afectados,
    v.empleados_con_voids,
    v.locaciones_con_voids,
    v.valor_total_voided,
    v.valor_prom_void,
    v.valor_max_void,
    COALESCE(s.total_items_vendidos, 0) AS total_items_vendidos,
    COALESCE(s.total_ventas, 0) AS total_ventas,
    COALESCE(s.total_checks, 0) AS total_checks_semana,
    COALESCE(s.locaciones_activas, 0) AS locaciones_activas,
    ROUND(v.valor_total_voided * 100.0 / NULLIF(s.total_ventas, 0), 2) AS pct_void_vs_ventas,
    ROUND(v.total_items_voided * 100.0 / NULLIF(v.total_items_voided + s.total_items_vendidos, 0), 2) AS pct_void_rate,
    te.employee_id AS top_empleado_voids,
    te.location_id AS top_empleado_locacion,
    te.valor_voided AS top_empleado_valor,
    tl.location_id AS top_locacion_voids,
    tl.valor_voided AS top_locacion_valor
FROM weekly_voids v
LEFT JOIN weekly_sales s ON v.semana_inicio = s.semana_inicio
LEFT JOIN top_employee te ON v.semana_inicio = te.semana_inicio
LEFT JOIN top_location tl ON v.semana_inicio = tl.semana_inicio
ORDER BY v.semana_inicio DESC;

In [0]:
%sql
-- 9. Scoring de riesgo consolidado por empleado x locacion (IQR multivariado)
--    Combina 4 senales activas con IQR + pesos ponderados redistribuidos (suman 100),
--    pesos: void_rate (33), pct_free_N/C (28), checks_espejo (22), valor_voided (17)
--    Las columnas night_voids y flag_night_voids se mantienen como informativas
--    (el dashboard depende de ellas; los umbrales IQR y niveles de riesgo no cambian).
CREATE OR REPLACE TABLE workspace.pipe.fraud_risk_score AS
WITH
-- Dimension 1+4: Void rate y valor voided por empleado x location
emp_voids AS (
    SELECT employee_id, location_id,
           SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) AS items_voided,
           SUM(CASE WHEN quantity >= 1.0 THEN 1 ELSE 0 END) AS items_vendidos,
           COUNT(*) AS total_items,
           ROUND(SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN sold_price ELSE 0 END), 2) AS valor_voided
    FROM workspace.applebees_clean.applebees_checks_items
    GROUP BY employee_id, location_id
    HAVING SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) > 0
),
emp_rates AS (
    SELECT employee_id, location_id,
           items_voided, items_vendidos, total_items, valor_voided,
           ROUND(items_voided * 100.0 / total_items, 2) AS pct_void_rate
    FROM emp_voids
),
-- Dimension 2: N/C rate por empleado x locacion
emp_nc AS (
    SELECT ci.employee_id, ci.location_id,
           COUNT(*) AS total_items_nc,
           SUM(CASE WHEN UPPER(i.item_name) LIKE '%N/C%' THEN 1 ELSE 0 END) AS free_items
    FROM workspace.applebees_clean.applebees_checks_items ci
    JOIN workspace.applebees_clean.applebees_items i ON ci.item_id = i.item_id
    GROUP BY ci.employee_id, ci.location_id
    HAVING SUM(CASE WHEN UPPER(i.item_name) LIKE '%N/C%' THEN 1 ELSE 0 END) > 0
),
-- Dimension 3: Patron espejo por empleado x locacion
emp_mirror AS (
    SELECT employee_id, location_id, COUNT(*) AS checks_con_espejo
    FROM workspace.pipe.fraud_voids_vs_sold
    WHERE flag_patron_espejo = 1
    GROUP BY employee_id, location_id
),
-- Dimension 5: Voids nocturnos (00:00-04:00) por empleado x locacion
emp_night AS (
    SELECT employee_id, location_id,
           SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) AS night_voids
    FROM workspace.applebees_clean.applebees_checks_items
    WHERE HOUR(order_time) BETWEEN 0 AND 4
    GROUP BY employee_id, location_id
    HAVING SUM(CASE WHEN quantity < 1.0 AND sold_price > 0 THEN 1 ELSE 0 END) > 0
),
-- Combinar todas las dimensiones
all_signals AS (
    SELECT v.employee_id, v.location_id,
           v.pct_void_rate,
           COALESCE(nc.free_items * 100.0 / nc.total_items_nc, 0) AS pct_free,
           COALESCE(m.checks_con_espejo, 0) AS checks_con_espejo,
           v.valor_voided,
           COALESCE(nv.night_voids, 0) AS night_voids
    FROM emp_rates v
    LEFT JOIN emp_nc nc ON v.employee_id = nc.employee_id AND v.location_id = nc.location_id
    LEFT JOIN emp_mirror m ON v.employee_id = m.employee_id AND v.location_id = m.location_id
    LEFT JOIN emp_night nv ON v.employee_id = nv.employee_id AND v.location_id = nv.location_id
),
-- IQR por dimension
iqr_stats AS (
    SELECT
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY pct_void_rate) AS q1_void,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY pct_void_rate) AS q3_void,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY pct_free) AS q1_free,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY pct_free) AS q3_free,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY checks_con_espejo) AS q1_mirror,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY checks_con_espejo) AS q3_mirror,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY valor_voided) AS q1_monto,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY valor_voided) AS q3_monto,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY night_voids) AS q1_night,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY night_voids) AS q3_night
    FROM all_signals
),
scored AS (
    SELECT
        s.employee_id, s.location_id,
        s.pct_void_rate, ROUND(s.pct_free, 2) AS pct_free,
        s.checks_con_espejo, s.valor_voided, s.night_voids,
        CASE WHEN s.pct_void_rate > i.q3_void + 1.5 * (i.q3_void - i.q1_void) THEN 1 ELSE 0 END AS flag_void_rate,
        CASE WHEN s.pct_free > i.q3_free + 1.5 * (i.q3_free - i.q1_free) THEN 1 ELSE 0 END AS flag_free_nc,
        CASE WHEN s.checks_con_espejo > i.q3_mirror + 1.5 * (i.q3_mirror - i.q1_mirror) THEN 1 ELSE 0 END AS flag_espejo,
        CASE WHEN s.valor_voided > i.q3_monto + 1.5 * (i.q3_monto - i.q1_monto) THEN 1 ELSE 0 END AS flag_monto,
        CASE WHEN s.night_voids > i.q3_night + 1.5 * (i.q3_night - i.q1_night) THEN 1 ELSE 0 END AS flag_night_voids,
        (CASE WHEN s.pct_void_rate > i.q3_void + 1.5 * (i.q3_void - i.q1_void) THEN 1 ELSE 0 END * 33 +
         CASE WHEN s.pct_free > i.q3_free + 1.5 * (i.q3_free - i.q1_free) THEN 1 ELSE 0 END * 28 +
         CASE WHEN s.checks_con_espejo > i.q3_mirror + 1.5 * (i.q3_mirror - i.q1_mirror) THEN 1 ELSE 0 END * 22 +
         CASE WHEN s.valor_voided > i.q3_monto + 1.5 * (i.q3_monto - i.q1_monto) THEN 1 ELSE 0 END * 17) AS risk_score
    FROM all_signals s
    CROSS JOIN iqr_stats i
)
SELECT
    employee_id, location_id,
    pct_void_rate, pct_free, checks_con_espejo, valor_voided, night_voids,
    flag_void_rate, flag_free_nc, flag_espejo, flag_monto, flag_night_voids,
    risk_score,
    CASE
        WHEN risk_score >= 60 THEN 'CRITICO'
        WHEN risk_score >= 40 THEN 'ALTO'
        WHEN risk_score >= 20 THEN 'MEDIO'
        ELSE 'BAJO'
    END AS nivel_riesgo
FROM scored
ORDER BY risk_score DESC, valor_voided DESC;

In [0]:
%sql
INSERT INTO workspace.pipe.fraud_pipeline_run_log
SELECT COALESCE(MAX(run_id), 0) + 1, CURRENT_TIMESTAMP(), 'SUCCESS', 9
FROM workspace.pipe.fraud_pipeline_run_log;

SELECT 'Pipeline succesful. 9 tables upgraded in workspace.pipe' AS mensaje;

In [0]:
# Verificacion rapida: mostrar conteos de cada tabla de fraude generada
tables = [
    "fraud_voids_by_employee",
    "fraud_voids_by_location",
    "fraud_voids_by_hour",
    "fraud_top_voided_items",
    "fraud_checks_most_voids",
    "fraud_voids_vs_sold",
    "fraud_free_items_suspicious",
    "fraud_weekly_summary",
    "fraud_risk_score"
]

print("=" * 60)
print("RESUMEN DE TABLAS DE FRAUDE GENERADAS")
print("=" * 60)

for t in tables:
    try:
        count = spark.sql(f"SELECT COUNT(*) AS cnt FROM workspace.pipe.{t}").collect()[0]["cnt"]
        print(f"  {t}: {count} filas")
    except Exception as e:
        print(f"  {t}: ERROR - {e}")

print("\nPipeline completado.")